In [1]:
import os
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from utils8 import AudioCNN
from utils8.data import AudioDataset, get_dataloader

# Get Data
path = os.path.join('Data', 'Digits')
classes = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
dataset = AudioDataset(path, classes)

# Data Split and Subsets
idx = list(range(len(dataset)))
labels = dataset.labels
train_val_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)

train_val_set = Subset(dataset, train_val_idx)
test_set = Subset(dataset, test_idx)

# Opruna Study

In [2]:
import os
from torch.utils.tensorboard import SummaryWriter
from utils8.dir_managment import clean_dir

writer_path = os.path.join('runs', 'optuna')
clean_dir(writer_path)

os.makedirs(writer_path, exist_ok=True)
writer = SummaryWriter(writer_path)

Cleaning existing files at runs/optuna...


In [3]:
import numpy as np
import optuna
from functools import partial

from torch import nn
from torch import optim
from torch.utils.data import TensorDataset
from sklearn.model_selection import KFold

from utils8.training import *
from utils8.AudioCNN import AudioCNN

N_EPOCHS = 30
optuna.logging.set_verbosity(optuna.logging.WARNING) # No Reason to se results as we have tensorboard
def objective_nn(trial:optuna.Trial, writer:SummaryWriter):
    global N_EPOCHS, train_val_set

    # Network Params
    lr = trial.suggest_float("lr", 1e-5, 5e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 1e-5, 5e-1, log=True)
    patience = trial.suggest_int("patience", 2, 10)

    fold_loss = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_val_idx)):

        # Loaders
        train_loader, val_loader = get_train_loaders(train_val_set, train_idx=train_idx, val_idx=val_idx, batch_size=32)

        # Model, Optimizer, Criterion
        model = AudioCNN(dropout_rate=dropout_rate)
        criterion = nn.CrossEntropyLoss() # nn.MSELoss (better if we choose ordinal encoding)
        optimizer = optim.Adam(lr=lr, params=model.parameters(), weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=patience)

        # Fold Training
        loss = train_one_fold(fold, model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs=N_EPOCHS, write_model_dir=None, writer=None)
        fold_loss.append(loss)

    avg_loss = np.mean(fold_loss)
    # Writing Params
    writer.add_hparams(
        hparam_dict={
        'trial': trial.number,
        'lr': lr,
        'patience': patience,
        'weight_decay': weight_decay,
        'dropout_rate': dropout_rate,
    },
    metric_dict={'hparam/mean_loss': avg_loss})
    writer.flush()


    return avg_loss

objective = partial(objective_nn, writer=writer)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=25, show_progress_bar=True)

  0%|          | 0/25 [00:00<?, ?it/s]

In [4]:
# tensorboard --logdir Lab8/runs/optuna

# Results

In [5]:
import pandas as pd
trials = study.trials_dataframe().sort_values(by=['value'], ascending=True)

trials.head(5)

,number,value,datetime_start,datetime_complete,duration,params_dropout_rate,params_lr,params_patience,params_weight_decay,state
20,20,0.500792,2026-05-20 07:27:05.804786,2026-05-20 08:21:50.843694,0 days 00:54:45.038908,0.008297,0.001607,5,0.000143,COMPLETE
21,21,0.514238,2026-05-20 08:21:50.846912,2026-05-20 09:16:40.114468,0 days 00:54:49.267556,0.010482,0.001545,5,0.000138,COMPLETE
24,24,0.551658,2026-05-20 11:07:16.834302,2026-05-20 12:08:31.918352,0 days 01:01:15.084050,0.014147,0.001752,5,0.000037,COMPLETE
16,16,0.648541,2026-05-20 03:45:57.568698,2026-05-20 04:41:27.957551,0 days 00:55:30.388853,0.004735,0.000451,3,0.000044,COMPLETE
10,10,0.704174,2026-05-19 21:23:32.511006,2026-05-19 22:22:24.762476,0 days 00:58:52.251470,0.395976,0.001647,2,0.000012,COMPLETE


In [6]:
from datetime import datetime

now = datetime.now()
formatted_date = now.strftime("%d_%m_%Y(%H:%M)")

path = os.path.join('optuna_results', f'trial_{formatted_date}.csv')
os.makedirs('optuna_results', exist_ok=True)
trials.to_csv(path, index=False)